# 00 - Configuration

Shared configuration, paths, parameters, theme, and vocabularies.
Every downstream notebook starts with:

    %run 00_config.ipynb

Provides the `cfg` object (paths, parameters, firm metadata), the plotting
theme, the controlled vocabularies, and the corpus loader function.

In [1]:
# Install dependencies. Run once then comment out.
%pip install -q sentence-transformers vaderSentiment statsmodels
%pip install -q scikit-learn thefuzz rapidfuzz python-Levenshtein
%pip install -q pandas numpy matplotlib seaborn scipy doubleml tqdm
%pip install -q pdfplumber pdfminer.six psutil openpyxl
%pip install -q python-doctr[torch]

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Imports and logging setup.
import os, re, gc, sys, json, time, logging, warnings
from pathlib import Path
from typing import Optional
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
from IPython.display import display, HTML, Markdown

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s -- %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler()],
)
log = logging.getLogger("VERIS")

In [3]:
# Project root resolution. Set MY_PROJECT_ROOT manually if auto-detection fails.
MY_PROJECT_ROOT = None

def _resolve_root(override):
    if override is not None:
        p = Path(str(override))
        if p.exists():
            return p
        raise FileNotFoundError(f"MY_PROJECT_ROOT does not exist: {override}")
    skip = {"src", "notebooks", "scripts", "code", "nb", ".git"}
    candidate = Path.cwd()
    for _ in range(10):
        if candidate.name.lower() not in skip:
            if (candidate / "data" / "raw" / "reports").exists():
                return candidate
            if (candidate / "data").exists() and (candidate / "outputs").exists():
                return candidate
        parent = candidate.parent
        if parent == candidate:
            break
        candidate = parent
    warnings.warn(f"Project root auto-detection failed. Using cwd: {Path.cwd()}")
    return Path.cwd()

_ROOT = _resolve_root(MY_PROJECT_ROOT)

In [4]:
# Configuration class. Canonical paths, parameters, and firm metadata.
# Parameters are grouped by tier of defensibility:
#   Tier A - published standards / facts (CSRD 2021, ALPHA 0.05, company names)
#   Tier B - calibrated deliberate choices, each justified by a short comment
class Config:
    PROJECT_ROOT = _ROOT

    # Input directories
    DATA_RAW     = PROJECT_ROOT / "data" / "raw"
    PDF_DIR      = DATA_RAW / "reports"
    CT_DATA_DIR  = DATA_RAW / "climate_trace" / "DATA"

    # Intermediate directories
    TEXT_DIR     = PROJECT_ROOT / "data" / "processed" / "text"

    # Output directories
    OUT_DIR      = PROJECT_ROOT / "outputs"
    CSV_DIR      = OUT_DIR / "csv"
    FIGURES_DIR  = OUT_DIR / "figures"
    LOG_DIR      = PROJECT_ROOT / "logs"

    # Canonical CSV output paths
    EXTRACTION_AUDIT_CSV    = CSV_DIR / "extraction_audit.csv"
    FORENSIC_CSV            = CSV_DIR / "forensic_features.csv"
    SBERT_DRIFT_CSV         = CSV_DIR / "sbert_drift.csv"
    CT_PANEL_CSV            = CSV_DIR / "ct_emissions.csv"
    CT_TRAJECTORY_CSV       = CSV_DIR / "ct_emissions_full_trajectory.csv"
    VERIS_SCORES_CSV        = CSV_DIR / "veris_scores.csv"
    MASTER_PANEL_CSV        = CSV_DIR / "master_panel.csv"
    GRANGER_CSV             = CSV_DIR / "granger_results.csv"
    CAUSAL_CSV              = CSV_DIR / "causal_results.csv"
    LEADERBOARD_CSV         = CSV_DIR / "leaderboard.csv"

    # Dashboard CSVs (veris_* prefix, consumed by report and Streamlit app)
    VERIS_MASTER_CSV        = CSV_DIR / "veris_master.csv"
    VERIS_EMISSIONS_CSV     = CSV_DIR / "veris_emissions.csv"
    VERIS_CAUSAL_CSV        = CSV_DIR / "veris_causal.csv"
    VERIS_LEADERBOARD_CSV   = CSV_DIR / "veris_leaderboard.csv"
    VERIS_SENSITIVITY_CSV   = CSV_DIR / "veris_sensitivity.csv"
    VERIS_AHP_WEIGHTS_CSV   = CSV_DIR / "veris_ahp_weights.csv"
    VERIS_LDA_KEYWORDS_CSV  = CSV_DIR / "veris_lda_keywords.csv"
    VERIS_QUALITATIVE_CSV   = CSV_DIR / "veris_qualitative_summary.csv"
    VALIDATION_RESULTS_CSV  = CSV_DIR / "validation_results.csv"

    # --- Pipeline parameters (Tier A and Tier B) -------------------------
    TARGET_YEARS        = list(range(2014, 2025))     # Tier A: study window
    PAGE_FILTER_LIMIT   = 90                          # Tier B: first 90 pages capture executive summary and performance sections, exclude appendices
    SBERT_MODEL         = "all-mpnet-base-v2"         # Tier B: 768-dim embeddings, Reimers and Gurevych (2019) leaderboard top-tier
    SBERT_CHAR_CAP      = 150_000                     # Tier B: empirical RAM feasibility limit for mpnet on consumer hardware
    SBERT_BATCH_SZ      = 8                           # Tier B: hardware-dependent batch size
    OCR_CHAR_THRESHOLD  = 500                         # Tier B: below this, Layer N text is assumed incomplete and escalates to Layer N+1
    OCR_DPI             = 150                         # Tier B: rendering resolution for docTR page rasterisation
    LDA_N_TOPICS        = 7                           # Tier B: aligned with GRI universal standards topic count for sustainability disclosure
    LDA_MAX_FEATURES    = 1000                        # Tier B: standard upper bound for LDA vocabulary (Blei, Ng and Jordan, 2003)
    LDA_RANDOM_STATE    = 42                          # Tier A: reproducibility seed
    JACCARD_NGRAM       = 3                           # Tier B: character trigrams, standard morphologically-sensitive configuration
    VADER_CHUNK_SIZE    = 500                         # Tier B: 500-word chunks balance context and sentence-level granularity
    CSRD_YEAR           = 2021                        # Tier A: Corporate Sustainability Reporting Directive adoption year
    ALPHA               = 0.05                        # Tier A: conventional significance threshold
    DML_FOLDS           = 5                           # Tier B: 5-fold cross-fitting per Chernozhukov et al. (2018)
    VERIS_THRESHOLD     = None                        # Computed from corpus median at runtime in nb04

    # Pilot subsample for facility-direct causal estimate (post-2021 only)
    # Tier B: 4 firms with highest CT facility-direct coverage density post-2021, minimising measurement error.
    PILOT_FIRMS         = ["RioTinto", "ConocoPhillips", "Equinor", "Chevron"]

    # Per-company fuzzy thresholds for CT ownership matching.
    # Tier B: default 62 percent, with overrides calibrated by manual validation against known CT facility ownership records (Phase D).
    CT_FUZZY_DEFAULT    = 62
    CT_FUZZY_OVERRIDES  = {
        "BP": 75, "Eni": 72, "RioTinto": 70, "Shell": 68, "Glencore": 65,
    }

    # 12 target firms with sector and HQ metadata (Tier A: ground truth).
    COMPANIES = {
        "BP":             {"sector": "Oil & Gas", "hq": "GBR", "eu": True},
        "Shell":          {"sector": "Oil & Gas", "hq": "GBR", "eu": True},  # UK HQ post-Brexit; Shell PLC retains dual-listing on LSE/Euronext Amsterdam
        #   and is subject to EU NFRD/CSRD via Dutch-registered subsidiary Shell Nederland.
        "Eni":            {"sector": "Oil & Gas", "hq": "ITA", "eu": True},
        "Equinor":        {"sector": "Oil & Gas", "hq": "NOR", "eu": True},
        "TotalEnergies":  {"sector": "Oil & Gas", "hq": "FRA", "eu": True},
        "Repsol":         {"sector": "Oil & Gas", "hq": "ESP", "eu": True},
        "ExxonMobil":     {"sector": "Oil & Gas", "hq": "USA", "eu": False},
        "Chevron":        {"sector": "Oil & Gas", "hq": "USA", "eu": False},
        "ConocoPhillips": {"sector": "Oil & Gas", "hq": "USA", "eu": False},
        "Glencore":       {"sector": "Mining",    "hq": "CHE", "eu": False},
        "RioTinto":       {"sector": "Mining",    "hq": "GBR", "eu": True},  # GBR post-Brexit; retained eu=True because Rio Tinto PLC reports under
        #   UK mandatory climate-related disclosure (TCFD-aligned, 2022+) and has
        #   substantial EU-operating subsidiaries subject to CSRD Article 19a.
        #   Sensitivity: re-running with eu=False does not change any headline result.
        "Unilever":       {"sector": "Consumer",  "hq": "GBR", "eu": True},
    }
    EU_FIRMS  = {k for k, v in COMPANIES.items() if v["eu"]}
    ALL_FIRMS = set(COMPANIES.keys())

    # Firm name aliases for fuzzy matching against Climate TRACE owner strings
    FIRM_ALIASES = {
        "BP":             ["BP", "BP PLC", "British Petroleum", "BP Exploration", "BP Oil", "ARCO", "BP America"],
        "Shell":          ["Shell", "Royal Dutch Shell", "Shell PLC", "Shell Nederland", "Shell Oil", "Shell Chemicals"],
        "Eni":            ["Eni", "ENI SpA", "Eni S.p.A", "Agip", "Versalis", "Saipem"],
        "Equinor":        ["Equinor", "Equinor ASA", "Statoil", "StatoilHydro"],
        "TotalEnergies":  ["TotalEnergies", "Total", "Total SE", "TotalEnergies SE", "Total S.A.", "TOTAL"],
        "Repsol":         ["Repsol", "Repsol SA", "Repsol YPF", "Repsol E&P"],
        "ExxonMobil":     ["ExxonMobil", "Exxon Mobil", "Exxon", "Mobil", "ExxonMobil Corp", "Imperial Oil"],
        "Chevron":        ["Chevron", "Chevron Corp", "ChevronTexaco", "Texaco", "Chevron U.S.A"],
        "ConocoPhillips": ["ConocoPhillips", "Conoco", "Phillips 66", "Burlington Resources"],
        "Glencore":       ["Glencore", "Glencore PLC", "Xstrata"],
        "RioTinto":       ["Rio Tinto", "RioTinto", "Rio Tinto PLC", "Rio Tinto Ltd"],
        "Unilever":       ["Unilever", "Unilever PLC", "Unilever N.V.", "Hindustan Unilever"],
    }

    # Climate TRACE sectors to load (Tier A: defined by CT v5.4.1 taxonomy).
    CT_SECTORS = [
        "oil-and-gas-production", "oil-and-gas-refining", "oil-and-gas-transport",
        "coal-mining", "bauxite-mining", "iron-mining", "copper-mining",
        "aluminum", "cement", "chemicals", "iron-and-steel",
        "petrochemical-steam-cracking", "pulp-and-paper",
        "electricity-generation",
        "domestic-aviation", "international-aviation",
    ]

    @classmethod
    def ensure_dirs(cls):
        for d in [cls.TEXT_DIR, cls.CSV_DIR, cls.FIGURES_DIR, cls.LOG_DIR]:
            d.mkdir(parents=True, exist_ok=True)
        log.info(f"Project root: {cls.PROJECT_ROOT.resolve()}")

    @classmethod
    def validate(cls):
        checks = {
            "data/raw/reports":    cls.PDF_DIR,
            "data/processed/text": cls.TEXT_DIR,
            "outputs/csv":         cls.CSV_DIR,
            "outputs/figures":     cls.FIGURES_DIR,
            "climate_trace/DATA":  cls.CT_DATA_DIR,
        }
        for label, path in checks.items():
            status = "OK" if path.exists() else "MISSING"
            log.info(f"  [{status}] {label}: {path}")

cfg = Config()
cfg.ensure_dirs()
cfg.validate()

18:43:45 [INFO] VERIS -- Project root: E:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20
18:43:45 [INFO] VERIS --   [OK] data/raw/reports: e:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20\data\raw\reports
18:43:45 [INFO] VERIS --   [OK] data/processed/text: e:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20\data\processed\text
18:43:45 [INFO] VERIS --   [OK] outputs/csv: e:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20\outputs\csv
18:43:45 [INFO] VERIS --   [OK] outputs/figures: e:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20\outputs\figures
18:43:45 [INFO] VERIS --   [OK] climate_trace/DATA: e:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20\data\raw\climate_trace\DATA


In [5]:
# Plot theme. Clean white sustainability palette used across all figures.
BG       = "white"
SURFACE  = "#f8fafb"
BORDER   = "#dde3e8"
PRIMARY  = "#1a7340"
ACCENT   = "#2d9e5f"
LIGHT    = "#52b788"
TEXT     = "#1a1a2e"
MUTED    = "#6b7280"
WARN     = "#e63946"
AMBER    = "#f4a261"

QUADRANT_COLORS = {
    "1 - Genuine improvement": "#1a7340",
    "2 - Greenwashing signal": "#e63946",
    "3 - Greenhushing":        "#f4a261",
    "4 - Stagnant":            "#6b7280",
    "Insufficient data":       "#c9d0d8",
}
SECTOR_PALETTE = {
    "Oil & Gas": "#2d9e5f",
    "Mining":    "#f4a261",
    "Consumer":  "#74b3ce",
}
FIRM_COLORS = {
    "BP":             "#006400", "Shell":          "#ffd700",
    "Eni":            "#0057a8", "Equinor":        "#e8003d",
    "TotalEnergies":  "#f05a22", "Repsol":         "#003da5",
    "ExxonMobil":     "#da0000", "Chevron":        "#003087",
    "ConocoPhillips": "#cc0000", "Glencore":       "#5c5c5c",
    "RioTinto":       "#c8102e", "Unilever":       "#1f3a8f",
}

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": SURFACE,
    "axes.edgecolor": BORDER, "axes.labelcolor": TEXT,
    "axes.grid": True, "grid.color": BORDER, "grid.alpha": 0.6,
    "grid.linestyle": "--", "xtick.color": TEXT, "ytick.color": TEXT,
    "text.color": TEXT, "legend.facecolor": SURFACE,
    "legend.edgecolor": BORDER, "font.family": "sans-serif",
    "figure.dpi": 120,
})

def save_fig(fig, name):
    """Save figure to FIGURES_DIR at 300 DPI with tight bounding box."""
    cfg.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    path = cfg.FIGURES_DIR / name
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor=BG)
    log.info(f"Saved figure: {path.name}")
    plt.show()

def save_fig_data(df, name):
    """Save the DataFrame that produced a figure, so viz can be re-created in Canva or elsewhere."""
    cfg.CSV_DIR.mkdir(parents=True, exist_ok=True)
    path = cfg.CSV_DIR / name
    df.to_csv(path, index=False)
    log.info(f"Saved figure data: {path.name}")

In [6]:
# Controlled vocabularies.
# Loughran-McDonald Litigious category for controversy density (Loughran & McDonald, 2016).
# Sourced from the LM Master Dictionary 1993-2025 (Litigious column != 0).
# 905 terms: extracted programmatically from the official 2020 LM release
# (matches Phase C vocabulary exactly; one term superset over pysentiment2 LM 2018).
# Self-contained - no external package dependency for vocabulary.
CONTROVERSY_TERMS = [
    "abovementioned", "abrogate", "abrogated", "abrogates", "abrogating", "abrogation",
    "abrogations", "absolve", "absolved", "absolves", "absolving", "accession",
    "accessions", "acquirees", "acquirors", "acquit", "acquits", "acquittal",
    "acquittals", "acquittance", "acquittances", "acquitted", "acquitting", "addendums",
    "adjourn", "adjourned", "adjourning", "adjournment", "adjournments", "adjourns",
    "adjudge", "adjudged", "adjudges", "adjudging", "adjudicate", "adjudicated",
    "adjudicates", "adjudicating", "adjudication", "adjudications", "adjudicative", "adjudicator",
    "adjudicators", "adjudicatory", "admissibility", "admissible", "admissibly", "admission",
    "admissions", "affidavit", "affidavits", "affirmance", "affreightment", "aforedescribed",
    "aforementioned", "aforesaid", "aforestated", "aggrieved", "allegation", "allegations",
    "allege", "alleged", "allegedly", "alleges", "alleging", "amend",
    "amendable", "amendatory", "amended", "amending", "amendment", "amendments",
    "amends", "antecedent", "antecedents", "anticorruption", "antitrust", "anywise",
    "appeal", "appealable", "appealed", "appealing", "appeals", "appellant",
    "appellants", "appellate", "appellees", "appointor", "appurtenance", "appurtenances",
    "appurtenant", "arbitrability", "arbitral", "arbitrate", "arbitrated", "arbitrates",
    "arbitrating", "arbitration", "arbitrational", "arbitrations", "arbitrative", "arbitrator",
    "arbitrators", "arrearage", "arrearages", "ascendancy", "ascendant", "ascendants",
    "assertable", "assignation", "assignations", "assumable", "attest", "attestation",
    "attestations", "attested", "attesting", "attorn", "attorney", "attorneys",
    "attornment", "attorns", "bail", "bailed", "bailee", "bailees",
    "bailiff", "bailiffs", "bailment", "beneficial", "beneficiated", "beneficiation",
    "bona", "bonafide", "breach", "breached", "breaches", "breaching",
    "cedant", "cedants", "certiorari", "cession", "chattel", "chattels",
    "choate", "claim", "claimable", "claimant", "claimants", "claimholder",
    "claims", "clawbacks", "codefendant", "codefendants", "codicil", "codicils",
    "codification", "codifications", "codified", "codifies", "codify", "codifying",
    "collusion", "compensatory", "complainant", "complainants", "condemnor", "confiscatory",
    "consent", "consented", "consenting", "consents", "conservatorships", "constitution",
    "constitutional", "constitutionality", "constitutionally", "constitutions", "constitutive", "construe",
    "construed", "construes", "construing", "contestability", "contestation", "contract",
    "contracted", "contractholder", "contractholders", "contractible", "contractile", "contracting",
    "contracts", "contractual", "contractually", "contravene", "contravened", "contravenes",
    "contravening", "contravention", "contraventions", "controvert", "controverted", "controverting",
    "conveniens", "conveyance", "conveyances", "convict", "convicted", "convicting",
    "conviction", "convictions", "coterminous", "counsel", "counseled", "counselled",
    "counsels", "countersignor", "countersued", "countersuit", "countersuits", "court",
    "courtroom", "courts", "crime", "crimes", "criminal", "criminality",
    "criminalize", "criminalizing", "criminally", "criminals", "crossclaim", "crossclaims",
    "decedent", "decedents", "declarant", "decree", "decreed", "decreeing",
    "decrees", "defalcation", "defalcations", "defeasance", "defeasances", "defease",
    "defeased", "defeasement", "defeases", "defeasing", "defectively", "defendable",
    "defendant", "defendants", "deference", "delegable", "delegatable", "delegatee",
    "delegees", "demurred", "demurrer", "demurrers", "demurring", "demurs",
    "depose", "deposed", "deposes", "deposing", "deposition", "depositional",
    "depositions", "derogate", "derogated", "derogates", "derogating", "derogation",
    "derogations", "designator", "desist", "detainer", "devisees", "disaffiliation",
    "disaffirm", "disaffirmance", "disaffirmed", "disaffirms", "dispositive", "dispossession",
    "dispossessory", "distraint", "distributee", "distributees", "docket", "docketed",
    "docketing", "dockets", "donees", "duly", "ejectment", "encumber",
    "encumbered", "encumbering", "encumbers", "encumbrance", "encumbrancer", "encumbrancers",
    "encumbrances", "endorsee", "enforceability", "enforceable", "enforceably", "escheat",
    "escheated", "escheatment", "escrowing", "estoppel", "evidential", "evidentiary",
    "exceedance", "exceedances", "exceedences", "excised", "exculpate", "exculpated",
    "exculpates", "exculpating", "exculpation", "exculpations", "exculpatory", "executor",
    "executors", "executory", "executrices", "executrix", "executrixes", "extracontractual",
    "extracorporeal", "extrajudicial", "facie", "facto", "felonies", "felonious",
    "felony", "fide", "forbade", "forbear", "forbearance", "forbearances",
    "forbearing", "forbears", "forebear", "forebearance", "forebears", "forfeitability",
    "forfeitable", "forthwith", "forwhich", "fugitive", "fugitives", "furtherance",
    "grantor", "grantors", "henceforth", "henceforward", "hereafter", "hereby",
    "hereditaments", "herefor", "herefore", "herefrom", "herein", "hereinabove",
    "hereinafter", "hereinbefore", "hereinbelow", "hereof", "hereon", "hereto",
    "heretofore", "hereunder", "hereunto", "hereupon", "herewith", "herewithin",
    "immateriality", "impleaded", "inasmuch", "incapacity", "incarcerate", "incarcerated",
    "incarcerates", "incarcerating", "incarceration", "incarcerations", "inchoate", "incontestability",
    "incontestable", "indemnifiable", "indemnification", "indemnifications", "indemnified", "indemnifies",
    "indemnify", "indemnifying", "indemnitee", "indemnitees", "indemnities", "indemnitor",
    "indemnitors", "indemnity", "indict", "indictable", "indicted", "indicting",
    "indictment", "indictments", "indorsees", "inforce", "infraction", "infractions",
    "infringer", "injunction", "injunctions", "injunctive", "insofar", "interlocutory",
    "interpleader", "interpose", "interposed", "interposes", "interposing", "interposition",
    "interpositions", "interrogate", "interrogated", "interrogates", "interrogating", "interrogation",
    "interrogations", "interrogator", "interrogatories", "interrogators", "interrogatory", "intestacy",
    "intestate", "irrevocability", "irrevocable", "irrevocably", "joinder", "judicial",
    "judicially", "judiciaries", "judiciary", "juries", "juris", "jurisdiction",
    "jurisdictional", "jurisdictionally", "jurisdictions", "jurisprudence", "jurist", "jurists",
    "juror", "jurors", "jury", "juryman", "justice", "justices",
    "law", "lawful", "lawfully", "lawfulness", "lawmakers", "lawmaking",
    "laws", "lawsuit", "lawsuits", "lawyer", "lawyers", "legal",
    "legalese", "legality", "legalization", "legalizations", "legalize", "legalized",
    "legalizes", "legalizing", "legally", "legals", "legatee", "legatees",
    "legislate", "legislated", "legislates", "legislating", "legislation", "legislations",
    "legislative", "legislatively", "legislator", "legislators", "legislature", "legislatures",
    "libel", "libeled", "libelous", "libels", "licensable", "lienholders",
    "litigant", "litigants", "litigate", "litigated", "litigates", "litigating",
    "litigation", "litigations", "litigator", "litigators", "litigious", "litigiousness",
    "majeure", "mandamus", "mediate", "mediated", "mediates", "mediating",
    "mediation", "mediations", "mediator", "mediators", "misdemeanor", "misfeasance",
    "mistrial", "mistrials", "moreover", "motions", "mutandis", "nolo",
    "nonappealable", "nonbreaching", "noncontingent", "noncontract", "noncontractual", "noncontributory",
    "nonfeasance", "nonfiduciary", "nonforfeitability", "nonforfeitable", "nonforfeiture", "nonguarantor",
    "noninfringement", "noninfringing", "nonjudicial", "nonjudicially", "nonjurisdictional", "nonseverable",
    "nonterminable", "nonusurious", "notarial", "notaries", "notarization", "notarizations",
    "notarize", "notarized", "notarizing", "notary", "notwithstanding", "novo",
    "nullification", "nullifications", "nullified", "nullifies", "nullify", "nullifying",
    "nullities", "nullity", "obligee", "obligees", "obligor", "obligors",
    "offense", "offeree", "offerees", "offeror", "offerors", "optionee",
    "optionees", "overrule", "overruled", "overrules", "overruling", "para",
    "pari", "passu", "patentee", "pecuniarily", "perjury", "permittee",
    "permittees", "perpetrate", "perpetrated", "perpetrates", "perpetrating", "perpetration",
    "personam", "petition", "petitioned", "petitioner", "petitioners", "petitioning",
    "petitions", "plaintiff", "plaintiffs", "pleading", "pleadings", "pleads",
    "pleas", "pledgee", "pledgees", "pledgor", "pledgors", "possessory",
    "postclosing", "postclosure", "postcontract", "postjudgment", "preamendment", "predecease",
    "predeceased", "predeceases", "predeceasing", "prehearing", "prejudice", "prejudiced",
    "prejudices", "prejudicial", "prejudicing", "prepetition", "presumptively", "pretrial",
    "prima", "privity", "probate", "probated", "probates", "probating",
    "probation", "probational", "probationary", "probationer", "probationers", "probations",
    "promulgate", "promulgated", "promulgates", "promulgating", "promulgation", "promulgations",
    "promulgator", "promulgators", "prorata", "proration", "prosecute", "prosecuted",
    "prosecutes", "prosecuting", "prosecution", "prosecutions", "prosecutor", "prosecutorial",
    "prosecutors", "proviso", "provisoes", "provisos", "punishable", "quitclaim",
    "quitclaims", "rata", "ratable", "ratably", "reargument", "rebut",
    "rebuts", "rebuttable", "rebuttably", "rebuttal", "rebuttals", "rebutted",
    "rebutting", "recordation", "recoupable", "recoupment", "recoupments", "recourse",
    "recourses", "rectification", "rectifications", "recusal", "recuse", "recused",
    "recuses", "recusing", "redact", "redacted", "redacting", "redaction",
    "redactions", "referenda", "referendum", "referendums", "refile", "refiled",
    "refiles", "refiling", "regulate", "regulated", "regulates", "regulating",
    "regulation", "regulations", "regulative", "regulator", "regulators", "regulatory",
    "rehear", "reheard", "rehearing", "rehearings", "releasees", "remand",
    "remanded", "remanding", "remands", "remediate", "remediated", "remediating",
    "remediation", "remediations", "remedied", "remised", "repledged", "replevin",
    "reprorated", "requester", "requestor", "reregulation", "rescind", "rescinded",
    "rescinding", "rescinds", "rescission", "rescissions", "restitutionary", "retendering",
    "retrocede", "retroceded", "retrocessionaires", "revocability", "revocation", "revocations",
    "ruling", "rulings", "sentenced", "sentencing", "sequestrator", "settlement",
    "settlements", "severability", "severable", "severally", "severance", "severances",
    "shall", "statute", "statutes", "statutorily", "statutory", "subclause",
    "subclauses", "subdocket", "subleasee", "subleasehold", "sublessors", "sublicensee",
    "sublicensor", "subparagraph", "subparagraphs", "subpoena", "subpoenaed", "subpoenas",
    "subrogated", "subrogation", "subtrust", "subtrusts", "sue", "sued",
    "sues", "suing", "summoned", "summoning", "summons", "summonses",
    "supersede", "supersedeas", "superseded", "supersedes", "superseding", "sureties",
    "surety", "tenantability", "terminable", "terminus", "testamentary", "testify",
    "testifying", "testimony", "thence", "thenceforth", "thenceforward", "thereafter",
    "thereat", "therefrom", "therein", "thereinafter", "thereof", "thereon",
    "thereover", "thereto", "theretofor", "theretofore", "thereunder", "thereunto",
    "thereupon", "therewith", "tort", "tortious", "tortiously", "torts",
    "transferor", "transferors", "unappealable", "unappealed", "unconstitutional", "unconstitutionality",
    "unconstitutionally", "uncontracted", "undefeased", "undischarged", "unencumber", "unencumbered",
    "unenforceability", "unenforceable", "unlawful", "unlawfully", "unlawfulness", "unremediated",
    "unstayed", "unto", "usurious", "usurp", "usurpation", "usurped",
    "usurping", "usurps", "usury", "vendee", "vendees", "verdict",
    "verdicts", "viatical", "violative", "voidable", "voided", "voiding",
    "warrantees", "warrantor", "whatever", "whatsoever", "whensoever", "whereabouts",
    "whereas", "whereat", "whereby", "wherefore", "wherein", "whereof",
    "whereon", "whereto", "whereunder", "whereupon", "wherewith", "whistleblowers",
    "whomever", "whomsoever", "whosoever", "wilful", "willful", "willfully",
    "willfulness", "witness", "witnesses", "writ", "writs"
]
LM_AVAILABLE = True
log.info(f"LM Litigious vocabulary loaded: {len(CONTROVERSY_TERMS)} terms (self-contained)")

# GRI 401-405 Social Topic Standards keyword window (GRI, 2016; 2018).
SOCIAL_KEYWORDS = [
    "employee", "employees", "workforce", "worker", "workers",
    "labour", "labor", "diversity", "inclusion", "equity",
    "health", "safety", "wellbeing", "fatality", "fatalities",
    "injury", "injuries", "training", "development",
    "human rights", "gender pay", "trade union", "collective",
]

# Ellen MacArthur Foundation Circular Economy Glossary 23-term core vocabulary.
CIRCULARITY_VOCAB = [
    "circular economy", "circularity", "closed loop", "recycled content",
    "recyclable", "recycling", "upcycling", "reuse", "remanufacturing",
    "waste reduction", "waste diversion", "zero waste", "material recovery",
    "extended producer responsibility", "take-back", "refurbishment",
    "biodegradable", "compostable", "sustainably sourced", "renewable material",
    "design for disassembly", "product stewardship", "life cycle",
]

18:43:45 [INFO] VERIS -- LM Litigious vocabulary loaded: 905 terms (self-contained)


In [7]:
# Corpus loader. Reads cached .txt files from data/processed/text/<Firm>/<year>.txt
def load_corpus_from_text(text_dir):
    """Return dict keyed (firm_name, year) -> text string, cap at SBERT_CHAR_CAP chars."""
    corpus = {}
    text_dir = Path(text_dir)
    if not text_dir.exists():
        log.error(f"Text cache not found: {text_dir}")
        return corpus
    for firm_dir in sorted(text_dir.iterdir()):
        if not firm_dir.is_dir():
            continue
        for txt_path in sorted(firm_dir.glob("*.txt")):
            try:
                year = int(txt_path.stem)
                if year not in cfg.TARGET_YEARS:
                    continue
                text = txt_path.read_text(encoding="utf-8", errors="ignore").strip()
                if len(text) < 200:
                    continue
                corpus[(firm_dir.name, year)] = text[:cfg.SBERT_CHAR_CAP]
            except ValueError:
                continue
    firms = {k[0] for k in corpus}
    log.info(f"Corpus loaded: {len(corpus)} documents from {len(firms)} firms")
    return corpus

log.info("Config loaded. Ready for downstream notebooks.")

18:43:45 [INFO] VERIS -- Config loaded. Ready for downstream notebooks.
